## BERT HOTELES

#### BARCELO

In [10]:
!pip install -q sentence-transformers bertopic gensim pandas scikit-learn umap-learn hdbscan nbformat spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 83.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

pd.set_option("display.max_colwidth", 200)

In [35]:
import pandas as pd

# Cargar el dataset del Hotel Silken
df_barcelo = pd.read_csv('comentarios_silken_puerta_valencia.csv')

# Extraer positivos y negativos con su etiqueta
pos = df_barcelo[['positivo']].dropna().rename(columns={'positivo': 'text'})
pos['sentiment'] = 'Positivo'

neg = df_barcelo[['negativo']].dropna().rename(columns={'negativo': 'text'})
neg['sentiment'] = 'Negativo'

# Combinar ambos en un solo corpus
df_mixed = pd.concat([pos, neg], ignore_index=True)
documents = df_mixed['text'].astype(str).tolist()

print(f"Total de documentos (Pos + Neg): {len(documents)}")
display(df_mixed.head())

Total de documentos (Pos + Neg): 1186


,text,sentiment
0,camas fenomenales,Positivo
1,amabilidad y colaboración en todo momento.,Positivo
2,"es la quinta vez que me alojo esta temporada y sigo con la misma impresión, es el mejor hotel de 4 estrellas de la zona, por ubicación y por calidad-precio. las habitaciones interiores son las más...",Positivo
3,la habitacion estaba muy bien y el restaurante es buenisimo,Positivo
4,ubicación y personal,Positivo


### EMBEDDINGS

In [36]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedding_model.encode(documents, show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Configuramos UMAP y HDBSCAN tal como indica el manual para asegurar un clustering de calidad.

In [75]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import spacy

# Initialize spaCy for Spanish stopwords
nlp = spacy.load('es_core_news_sm')

# Reducción de dimensionalidad y Clustering
umap_model = UMAP(n_neighbors=15, n_components=3, min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=20, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Vectorizador para limpiar palabras vacías en los resultados

# Cargamos las stopwords de spaCy
stopwords_es = list(nlp.Defaults.stop_words)

# Añadimos palabras que no nos interesen específicamente de hoteles para separar entre positivo o negativo
stopwords_es.extend(["hotel", "valencia", "habitacion"])

#Configuramos el vectorizador
# El token_pattern r"(?u)\b[a-zA-ZáéíóúÁÉÍÓÚñÑ]{2,}\b" obliga a que los tokens:
# - Tengan al menos 2 caracteres.
# - Solo contengan letras (incluyendo tildes y ñ), eliminando así los números.
vectorizer_model = CountVectorizer(
    stop_words=stopwords_es, #ademas de quitar las stopwords
    token_pattern=r"(?u)\b[a-zA-ZáéíóúÁÉÍÓÚñÑ]{2,}\b"
)

# Al crear el modelo, le pasamos este vectorizador
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual"
)

# Aquí puedes usar los documents originales sin problemas
topics, probs = topic_model.fit_transform(documents, embeddings)
df_mixed["topic"] = topics

¿Sabe separarlos? (Análisis de Resultados)

In [76]:
# Ver la distribución de sentimiento por cada tema encontrado
df_distribucion = df_mixed.groupby(['topic', 'sentiment']).size().unstack(fill_value=0)
df_distribucion['Total'] = df_distribucion.sum(axis=1)
df_distribucion['% Positivo'] = (df_distribucion['Positivo'] / df_distribucion['Total'] * 100).round(2)

print("Distribución de sentimiento por Topic (¿Hubo separación?):")
display(df_distribucion)

# Mostrar palabras clave de los temas con más "Negativos" y más "Positivos"
topic_info = topic_model.get_topic_info()
display(topic_info.head(10))

Distribución de sentimiento por Topic (¿Hubo separación?):


sentiment,Negativo,Positivo,Total,% Positivo
topic,,,,
-1,139,116,255,45.49
0,91,64,155,41.29
1,8,95,103,92.23
2,29,51,80,63.75
3,2,73,75,97.33
4,8,64,72,88.89
5,55,7,62,11.29
6,53,4,57,7.02
7,3,46,49,93.88


,Topic,Count,Name,Representation,Representative_Docs
0,-1,255,-1_habitación_recepción_personal_centro,"[habitación, recepción, personal, centro, baño, cama, ubicación, camas, minutos, limpieza]","[la ubicación a 15 minutos del centro., reservé una habitación triple y llamé el día anterior para pedir que me pusieran una cuna para el bebé pero me aseguraron que en la habitación era una habi..."
1,0,155,0_perfecto_general_correcto_precio,"[perfecto, general, correcto, precio, mano, aprobación, señal, precios, calidad, relación]","[todo perfecto, todo perfecto señal de aprobación con la mano, en general, todo perfecto]"
2,1,103,1_habitación_amplia_habitaciones_amplias,"[habitación, amplia, habitaciones, amplias, terraza, amplitud, ubicación, personal, tamaño, limpia]","[habitación, la habitación, la habitación]"
3,2,80,2_desayuno_cena_variedad_comida,"[desayuno, cena, variedad, comida, variado, cenas, buffet, espacio, calidad, completo]","[el desayuno, desayuno muy bien, desayuno bueno]"
4,3,75,3_desayuno_personal_completo_variado,"[desayuno, personal, completo, variado, amable, habitación, cena, calidad, rico, parking]","[la habitación tiene mucha luz y el desayuno era bastante completo. el personal super amable. el parking amplio, la habitación grande, limpia y muy cómodas las camas, el desayuno con variedad y el..."
5,4,72,4_cama_cómoda_baño_camas,"[cama, cómoda, baño, camas, cómodas, almohadas, habitación, amplia, limpio, comoda]","[la cama es muy cómoda., la cama y la habitación, personal muy amable, cama muy cómoda, habitación amplia y baño completo.]"
6,5,62,5_ducha_agua_temperatura_bañera,"[ducha, agua, temperatura, bañera, aire, funcionaba, acondicionado, caliente, frío, baño]","[el baño, la ducha no tragaba el agua, salia fuera y la regulación del agua caliente o fria un camarero del bar, mal servicio precio desayuno, carísimo, la cama y la ducha., la ducha. en desayuno.]"
7,6,57,6_cafetería_café_desayuno_precio,"[cafetería, café, desayuno, precio, cafetera, ofrecieron, servicio, bar, caro, comer]","[no poder tomar un pequeño desayuno en la cafetería. o contratas el desayuno buffet o tienes que salir del hotel., fue realmente muy decepcionante y umillante el trato de una trabajadora la cual n..."
8,7,49,7_excelente_cerca_personal_limpio,"[excelente, cerca, personal, limpio, cómodo, confortable, calidad, atención, ubicacion, precio]","[muy buen hotel relación calidad precio. habitación grande y cama confortable. excelente baño. buen trato por parte del personal. limpió y cuidado, el personal de primera. tuvimos un percance de s..."
9,8,43,8_ubicación_instalaciones_ubicacion_localización,"[ubicación, instalaciones, ubicacion, localización, entrada, personal, juniors, planeado, oscuros, situacion]","[instalaciones , personal , ubicación, las instalaciones y la ubicación, la ubicación y las instalaciones]"


In [77]:
# Mapa de distancias inter-tópicas (verás grupos de quejas vs grupos de elogios)
topic_model.visualize_topics()

In [54]:

# Gráfico de barras para comparar los términos de los temas
topic_model.visualize_barchart()

In [55]:
# Mapa de calor
topic_model.visualize_heatmap()